from ket import *
from IPython.display import display

from ket import __version__
print(f"Ket v{__version__}")

# Introdução
Computação Quântica - FSC7152

Primeiro, será apresentado a implementação do circuito SWAP Test isoladamente. Em seguida será mostrada a implementação do SWAP Test sob múltiplas iterações, a fim de gerar o resultado da probabilidade obtido após várias medidas, definidas pelo tamanho do máximo do erro. A seguir, uma série de exemplos são realizados.

## Seção 1: Implementando o SWAP Test

O objetivo é implementar a função SWAP Test, que será usada posteriormente para avaliar a similaridade entre duas mensagens de $n$ qubits.

O circuito do SWAP Test é definido como:


In [123]:
def swap_test(m1: Quant, m2: Quant, ancilla: Quant):
    with around(H, ancilla):
        for i in range(len(m1)):
            ctrl(ancilla, SWAP)(m1[i], m2[i])
    return measure(ancilla).value

## Seção 2: Implementando o Multiple SWAP Test

O SWAP Test retorna um resultado probabílistico. Os resultados de múltiplas medidas devem ser utilizados para se aproximar na probabilidade real de retorno da função SWAP Test. Para isso, introduzimos a próxima função, que além dos parâmetros do SWAP Test, recebe um float epslon, que indica a margem de erro.

In [124]:
from math import ceil
def multiple_swap_test(m1: Quant, m2: Quant, ancilla: Quant, epslon: float):
    m_zero = 0
    it = ceil(1 / epslon ** 2)
    for i in range(it):
        if (swap_test(m1, m2, ancilla)):
            m_zero += 1
    return m_zero / it

## Seção 3: Casos de Teste

Com as funções já implementadas, devemos agora verficar suas corretudes. Para isso, uma série de teste foram prepardos.

**SWAP Test:**

O código abaixo imprime o resultado da realização de um SWAP Test em dois estados completamente ortogonais. Verifica-se que ora obtém-se 0, ora obtém-se 1 como resposta.

In [133]:
n = 10
processo = Process()
ancilla = processo.alloc()
m1 = processo.alloc(n)
m2 = processo.alloc(n)

for i in range (n):
    H(m2[i])

result = swap_test(m1, m2, ancilla)

print(result)

1


O próximo exemplo avalia estados equivalentes. Nota-se que o resultado é sempre 0.

In [126]:
n = 10
processo = Process()
ancilla = processo.alloc()
m1 = processo.alloc(n)
m2 = processo.alloc(n)

for i in range (n):
    H(m2[i])

result = swap_test(m1, m2, ancilla)

print(result)

0


**Multiple SWAP Test:**

Agora, iremos testar o código que realiza múltiplas chamadas ao SWAP test, com objetivo de juntar todos os resultados obtidos para montar a probabilidade de medida. A seguir está o código de teste para dois estados ortogonais. Verificamos que quanto menor o epslon, mais o resultado se aproxima de 0.5, como esperado.

In [130]:
n = 10
epslon = 0.1

processo = Process()
ancilla = processo.alloc()
m1 = processo.alloc(n)
m2 = processo.alloc(n)

for i in range (n):
    H(m2[i])

result = multiple_swap_test(m1, m2, ancilla, epslon)

print(result)

0.47
